In [ ]:
import numpy as np, pandas as pd
from src.data import load_processed
from src.walks import build_adj, random_walk_counts, walk_graph
from src.baselines import popularity_fn, walk_fn, als_fn
from src.train import train_gnn, gnn_fn
from src.metrics import evaluate

D = load_processed()
SEEDS, Ks = (0, 1, 2), (10, 50)
BEST_T, BEST_D, BEST_LR = 20, 64, 1e-3          # substitua pelos valores escolhidos na validação
test_mask = (D['train_csr'] + D['val_csr']).tocsr()   # no teste, treino e validação já são conhecidos
target = D['test_csr']
pop = np.asarray(D['train_csr'].sum(0)).ravel()
indptr, indices = build_adj(D['train'], D['nU'], D['nA'])

rows, per_user, rec_store = [], {}, {}

def run(name, seed, fn, secs=0.0):
    s, pu, rc = evaluate(fn, test_mask, target, Ks)
    rows.append({'modelo': name, 'seed': seed, 'segundos_treino': secs, **s})
    per_user[(name, seed)] = pu; rec_store[(name, seed)] = rc

run('Popularidade', 0, popularity_fn(D['train_csr']))
for seed in SEEDS:
    C = random_walk_counts(indptr, indices, seed=seed)
    run('Caminhada (Pixie)', seed, walk_fn(C, D['nU'], pop))
    run('ALS', seed, als_fn(D['train_w_csr'], seed=seed))      # use os melhores parâmetros da validação
    ei, ew = walk_graph(C, BEST_T)
    for name, lw in (('GraphSAGE (pesos fixos)', False), ('GraphSAGE (pesos aprendidos)', True)):
        m, h, secs = train_gnn(D, ei, ew, learn_weights=lw, seed=seed, d=BEST_D, lr=BEST_LR)
        run(name, seed, gnn_fn(m, ei, ew), secs)
        if seed == 0:
            h.to_csv(f'results/curva_treino_{lw}.csv', index=False)

res = pd.DataFrame(rows)
res.to_csv('results/resultados_por_seed.csv', index=False)

In [ ]:
# tabela principal
cols = ['prec@10', 'rec@10', 'ndcg@10', 'cov@10', 'pop@10', 'ndcg@50', 'cov@50']
g = res.groupby('modelo')[cols]
tabela = g.mean().round(4).astype(str) + ' ± ' + g.std(ddof=0).fillna(0).round(4).astype(str)
tabela['segundos_treino'] = res.groupby('modelo')['segundos_treino'].mean().round(0)
tabela.to_csv('results/tabela_principal.csv')
print(tabela.to_markdown())

In [ ]:
# bootstrap pareado entre usuarios
def ic_bootstrap(a, b, n=1000, seed=0):
    rng = np.random.default_rng(seed); d = a - b
    m = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(n)]
    return d.mean(), np.percentile(m, [2.5, 97.5])

a = per_user[('GraphSAGE (pesos aprendidos)', 0)]['ndcg@10']
for ref in ('GraphSAGE (pesos fixos)', 'Caminhada (Pixie)', 'ALS', 'Popularidade'):
    diff, ci = ic_bootstrap(a, per_user[(ref, 0)]['ndcg@10'])
    print(f'vs {ref}: {diff:+.4f}  IC95% [{ci[0]:+.4f}, {ci[1]:+.4f}]')

In [ ]:
# graficos
import matplotlib.pyplot as plt
g2 = res.groupby('modelo')[cols].agg(['mean', 'std']).fillna(0)
ordem = list(g2.index)

# Fig. 1: qualidade (NDCG@10)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(ordem, g2[('ndcg@10', 'mean')], yerr=g2[('ndcg@10', 'std')], capsize=3)
ax.set(ylabel='NDCG@10', title='Qualidade da recomendação (teste)')
plt.xticks(rotation=20, ha='right'); plt.tight_layout(); plt.savefig('results/fig1_ndcg.png', dpi=200)

# Fig. 2: relevância x diversidade
fig, ax = plt.subplots(figsize=(5.5, 4))
for m in ordem:
    ax.errorbar(g2.loc[m, ('cov@10', 'mean')], g2.loc[m, ('ndcg@10', 'mean')],
                xerr=g2.loc[m, ('cov@10', 'std')], yerr=g2.loc[m, ('ndcg@10', 'std')], fmt='o', label=m)
ax.set(xlabel='Cobertura de catálogo @10', ylabel='NDCG@10', title='Relevância x diversidade')
ax.legend(fontsize=7); plt.tight_layout(); plt.savefig('results/fig2_relevancia_diversidade.png', dpi=200)

# Fig. 3: concentração das recomendações em poucos artistas
fig, ax = plt.subplots(figsize=(5.5, 4))
for m in ordem:
    rc = np.sort(rec_store[(m, 0)][10])[::-1]
    ax.plot(np.arange(1, len(rc) + 1) / len(rc), rc.cumsum() / rc.sum(), label=m)
ax.set(xlabel='Fração dos artistas (mais recomendados primeiro)',
       ylabel='Fração acumulada das recomendações', title='Concentração em itens populares')
ax.legend(fontsize=7); plt.tight_layout(); plt.savefig('results/fig3_concentracao.png', dpi=200)

# Fig. 4: curva de treino do GraphSAGE com pesos aprendidos (semente 0)
h = pd.read_csv('results/curva_treino_True.csv')
v = h.dropna(subset=['val_ndcg@10'])
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].plot(h['epoch'], h['loss']); ax[0].set(title='Perda BPR (treino)', xlabel='época')
ax[1].plot(v['epoch'], v['val_ndcg@10'], marker='o'); ax[1].set(title='NDCG@10 na validação', xlabel='época')
plt.tight_layout(); plt.savefig('results/fig4_treino.png', dpi=200)